# Análise Descritiva de Produtos - Capítulo 44 (NCM)

Este notebook contém uma análise descritiva robusta sobre os dados de itens de nota fiscal do Capítulo 44 do NCM, conforme extraído da base de dados. O objetivo é fornecer ferramentas para a limpeza, carregamento e análise inicial dos dados, focando em contagem, preço e representatividade, além de uma seção dedicada à validação e categorização de produtos via Expressões Regulares (Regex).

**Nota:** O arquivo `cap_44_cleaned.csv` foi gerado previamente, removendo as aspas duplas das colunas `prod_ncm_capitulo` e `prod_xprod` e mantendo a separação por tabulação (`\t`).

## 1. Configuração Inicial e Carregamento de Dados (Pandas)

In [ ]:
import pandas as pd
import re
import os

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Caminho para o arquivo CSV limpo (separado por tabulação)
# Certifique-se de que este arquivo esteja no mesmo diretório do notebook
FILE_PATH = 'cap_44_cleaned.csv'

print(f"Carregando dados com Pandas do arquivo: {FILE_PATH}")

# Carregamento com Pandas
# O arquivo é separado por tabulação ('\t') e não tem aspas duplas desnecessárias
try:
    # O parâmetro `decimal='.'` é importante para garantir que os números sejam lidos corretamente
    df_pandas = pd.read_csv(FILE_PATH, sep='\t', decimal='.')
    print(f"Dados carregados com sucesso. Total de linhas: {len(df_pandas)}")
    
    # Conversão de tipos para otimização e análise
    df_pandas['prod_vuntrib'] = pd.to_numeric(df_pandas['prod_vuntrib'], errors='coerce')
    df_pandas['prod_qtrib'] = pd.to_numeric(df_pandas['prod_qtrib'], errors='coerce')
    df_pandas['contagem'] = pd.to_numeric(df_pandas['contagem'], errors='coerce', downcast='integer')
    
    print("Informações iniciais do DataFrame:")
    df_pandas.info()
    print("\nPrimeiras 5 linhas:")
    print(df_pandas.head())
except FileNotFoundError:
    print(f"ERRO: Arquivo {FILE_PATH} não encontrado. Certifique-se de que ele está no diretório correto.")
except Exception as e:
    print(f"Ocorreu um erro durante o carregamento do Pandas: {e}")

## 2. Carregamento de Dados (PySpark)

In [ ]:
# Para usar PySpark, você precisará ter o ambiente configurado (Spark instalado e PySpark disponível).
# Se você não tiver o Spark instalado, comente esta seção ou instale-o.

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col
    
    # Inicializa a sessão Spark
    spark = SparkSession.builder \
        .appName("AnaliseCapitulo44") \
        .getOrCreate()
        
    print("Sessão Spark inicializada com sucesso.")
    
    # Carregamento com PySpark
    df_spark = spark.read.csv(
        FILE_PATH, 
        sep='\t', 
        header=True, 
        inferSchema=True
    )
    
    print(f"Dados carregados com PySpark. Total de linhas: {df_spark.count()}")
    print("Schema do DataFrame Spark:")
    df_spark.printSchema()
    df_spark.show(5, truncate=False)
    
    # Pare a sessão Spark quando terminar (opcional, mas boa prática)
    # spark.stop()
    
except ImportError:
    print("PySpark não está instalado. Para usar esta seção, instale com: pip install pyspark")
except Exception as e:
    print(f"Ocorreu um erro durante o carregamento do PySpark: {e}")

## 3. Análise de Contagem (`contagem`)

In [ ]:
# Ordenar os produtos por contagem (qtd) (tanto asc como desc)

print("### Top 50 Produtos por Contagem (Mais Frequentes) ###")
top_50_contagem = df_pandas.sort_values(by='contagem', ascending=False).head(50)
print(top_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

print("\n### Bottom 50 Produtos por Contagem (Menos Frequentes) ###")
# Produtos com contagem 1 (ocorrem apenas uma vez) são os menos frequentes
bottom_50_contagem = df_pandas.sort_values(by='contagem', ascending=True).head(50)
print(bottom_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

## 4. Análise de Preço (`prod_vuntrib` e `prod_qtrib`)

In [ ]:
print("### Top 50 Produtos por Valor Unitário Tributável (`prod_vuntrib`) ###")
top_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=False).head(50)
print(top_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

print("\n### Bottom 50 Produtos por Valor Unitário Tributável (`prod_vuntrib`) ###")
bottom_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=True).head(50)
print(bottom_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

print("\n### Top 50 Produtos por Quantidade Tributável (`prod_qtrib`) ###")
top_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=False).head(50)
print(top_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

print("\n### Bottom 50 Produtos por Quantidade Tributável (`prod_qtrib`) ###")
bottom_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=True).head(50)
print(bottom_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

## 5. Análise de Representatividade (Valor Total)

In [ ]:
# Criar colunas de representatividade (Valor Total = Valor Unitário * Contagem)
df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['contagem']
df_pandas['valor_total_qtrib'] = df_pandas['prod_qtrib'] * df_pandas['contagem']

print("### Top 50 Produtos por Valor Total Tributável (`valor_total_vuntrib`) ###")
top_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=False).head(50)
print(top_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

print("\n### Bottom 50 Produtos por Valor Total Tributável (`valor_total_vuntrib`) ###")
bottom_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=True).head(50)
print(bottom_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

print("\n### Top 50 Produtos por Quantidade Total Tributável (`valor_total_qtrib`) ###")
top_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=False).head(50)
print(top_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

print("\n### Bottom 50 Produtos por Quantidade Total Tributável (`valor_total_qtrib`) ###")
bottom_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=True).head(50)
print(bottom_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

## 6. Análise de Validação e Categorização por Regex (NCM 44.01)

In [ ]:
# O NCM 44.01 refere-se a 'Lenha em qualquer estado; madeira em estilhas ou em partículas; serragem, desperdícios e resíduos de madeira, mesmo aglomerados em briquetes, 'pellets' ou em formas semelhantes.'

# Foco: Criar regex manuais para identificar sequências de palavras-chave relacionadas ao 44.01

# Exemplo de Regex para Lenha e Madeira em Estilhas/Partículas
regex_lenha_estilha = r"(LENHA|MADEIRA.*(ESTILHA|PARTICULA|CAVACO)|SERRAGEM|BRIQUETE|PELLET)"

# Exemplo de Regex para Serragem e Resíduos
regex_serragem_residuo = r"(SERRAGEM|RESIDUO.*MADEIRA|PO.*MADEIRA)"

# Lista de tuplas: (Nome da Categoria, Regex)
categorias_regex = [
    ('4401_LENHA_ESTILHA', regex_lenha_estilha),
    ('4401_SERRAGEM_RESIDUO', regex_serragem_residuo),
    # Adicione mais categorias e regex conforme a necessidade de análise
    # Ex: ('4402_CARVAO', r"(CARVAO.*VEGETAL)")
]

def categorizar_produto(descricao):
    # Converte para maiúsculas para facilitar a correspondência
    descricao_upper = str(descricao).upper()
    
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'OUTROS_44'

# Aplica a função de categorização à coluna prod_xprod
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

print("### Distribuição de Produtos por Categoria Regex (Top 20) ###")
distribuicao_regex = df_pandas['categoria_regex'].value_counts().head(20)
print(distribuicao_regex)

print("\n### Produtos que Caíram na Categoria '4401_LENHA_ESTILHA' (Amostra) ###")
amostra_lenha = df_pandas[df_pandas['categoria_regex'] == '4401_LENHA_ESTILHA'].head(10)
print(amostra_lenha[['prod_xprod', 'contagem', 'categoria_regex']].reset_index(drop=True))

# Análise de Representatividade por Categoria Regex
print("\n### Representatividade por Categoria Regex (Valor Total Vuntrib) ###")
representatividade_regex = df_pandas.groupby('categoria_regex')['valor_total_vuntrib'].sum().sort_values(ascending=False)
print(representatividade_regex.head(20))

## 7. Resumo e Próximos Passos

As análises acima fornecem uma visão inicial da distribuição de contagem, preço e valor total dos produtos do Capítulo 44. A seção de Regex serve como um ponto de partida para a validação e categorização dos produtos, permitindo refinar as expressões para isolar subcategorias específicas do NCM (como 44.01, 44.02, etc.) e verificar se os produtos descritos (`prod_xprod`) realmente pertencem ao capítulo identificado.

**Sugestões de Próximos Passos:**
1. **Visualização de Dados:** Criar gráficos de barras para o Top 50 de Contagem e Valor Total.
2. **Refinamento do Regex:** Expandir a lista `categorias_regex` para cobrir outros subcapítulos do NCM 44 (e.g., 44.02 Carvão Vegetal, 44.03 Madeira em Bruto, etc.).
3. **Análise de Outliers:** Investigar os produtos com valores extremos de `prod_vuntrib` e `prod_qtrib`.